In [1]:
!pip install pdfplumber


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 98.5 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [4]:
from google.colab import files
import os

print("Selecione o arquivo PDF oficial da NR (ex: NR-13.pdf) para fazer o upload:")
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print(f"\n✅ Arquivo carregado com sucesso: {pdf_filename}")


Selecione o arquivo PDF oficial da NR (ex: NR-13.pdf) para fazer o upload:


Saving NR-05.pdf to NR-05.pdf
Saving NR-06.pdf to NR-06.pdf
Saving NR-10.pdf to NR-10.pdf
Saving NR-11.pdf to NR-11.pdf
Saving NR-12.pdf to NR-12.pdf
Saving NR-13.pdf to NR-13.pdf
Saving NR-20.pdf to NR-20.pdf
Saving NR-23.pdf to NR-23.pdf
Saving NR-33.pdf to NR-33.pdf
Saving NR-35.pdf to NR-35.pdf

✅ Arquivo carregado com sucesso: NR-05.pdf


In [5]:
import os
import re
import json
import zipfile
import pdfplumber
import glob
from google.colab import files
from dataclasses import dataclass, field, asdict

# 1. DETECTAR OS ARQUIVOS QUE VOCÊ JÁ SUBIU
# Tenta pegar os nomes do upload anterior; se não achar, busca na pasta atual do Colab
try:
    arquivos_pdf = list(uploaded.keys())
    # Garante que os arquivos realmente existem no disco
    arquivos_pdf = [f for f in arquivos_pdf if os.path.exists(f)]
except NameError:
    # Busca por qualquer arquivo PDF na pasta atual do Colab
    arquivos_pdf = glob.glob("*.pdf")

if not arquivos_pdf:
    print("❌ Nenhum arquivo PDF foi detectado. Por favor, certifique-se de que rodou a célula de upload com sucesso.")
else:
    print(f"📂 Encontrado(s) {len(arquivos_pdf)} arquivo(s) para processar: {arquivos_pdf}")

    # ================= DEFINIÇÃO DO PARSER =================
    @dataclass
    class Node:
        node_id: str
        document_id: str
        document_type: str
        chapter_number: str
        depth: int
        chapter_path: str
        title: str | None
        content: str
        has_table: bool = False
        references: list = field(default_factory=list)

    FOOTER_NOISE = {"Este texto não substitui o publicado no DOU"}

    ROMAN_ANEXO_RE = re.compile(r"^ANEXO\s+([IVXLCDM]+)(?:\s+DA\s+NR-?\s*(\d+))?$")
    GLOSSARIO_RE = re.compile(r"^GLOSS[ÁA]RIO\s*$", re.IGNORECASE)
    LETTERED_ITEM_RE = re.compile(r"^[a-z]\)\s")

    REF_NR_RE = re.compile(r"\bNR[\s-]?(\d{1,2})\b")
    REF_SUBITEM_RE = re.compile(r"subitem\s+(\d{1,2}(?:\.\d{1,4}){1,6})", re.IGNORECASE)
    REF_ANEXO_RE = re.compile(r"Anexo\s+([IVXLCDM]+)\s+desta\s+NR", re.IGNORECASE)

    def extract_lines(pdf_path: str) -> list[str]:
        with pdfplumber.open(pdf_path) as pdf:
            pages_text = [p.extract_text() or "" for p in pdf.pages]
        raw = "\n".join(pages_text)
        lines = [l.strip() for l in raw.split("\n")]
        return [l for l in lines if l and l not in FOOTER_NOISE]

    def make_heading_re(chapter_prefix: str) -> re.Pattern:
        return re.compile(rf"^({re.escape(chapter_prefix)}(?:\.\d{{1,4}}){{0,6}})\s+(\S.*)$")

    def is_section_header(text: str) -> bool:
        words = text.split()
        if len(words) > 8:
            return False
        if text.rstrip().endswith((".", ";", ":")):
            return False
        return True

    def extract_references(content: str, self_doc: str, node_lookup: dict) -> list[dict]:
        refs = []
        seen = set()
        for m in REF_NR_RE.finditer(content):
            num = m.group(1)
            target = f"NR-{int(num):02d}"
            if target == self_doc:
                continue
            key = ("doc", target)
            if key in seen:
                continue
            seen.add(key)
            refs.append({
                "target_document": target,
                "target_node": None,
                "relation": "referencia",
                "source": f"regex: menção direta a '{m.group(0)}' no texto"
            })
        for m in REF_SUBITEM_RE.finditer(content):
            num = m.group(1)
            target_node_id = f"{self_doc.replace('-', '')}_{num}"
            key = ("subitem", num)
            if key in seen:
                continue
            seen.add(key)
            refs.append({
                "target_document": self_doc,
                "target_node": target_node_id if target_node_id in node_lookup else None,
                "relation": "referencia",
                "source": f"regex: menção direta a 'subitem {num}' no texto"
            })
        for m in REF_ANEXO_RE.finditer(content):
            roman = m.group(1)
            key = ("anexo", roman)
            if key in seen:
                continue
            seen.add(key)
            refs.append({
                "target_document": self_doc,
                "target_node": f"{self_doc.replace('-', '')}_ANEXO_{roman}",
                "relation": "referencia",
                "source": f"regex: menção direta a 'Anexo {roman} desta NR' no texto"
            })
        return refs

    def parse_nr(pdf_path: str, document_id: str, chapter_prefix: str) -> list[Node]:
        lines = extract_lines(pdf_path)
        heading_re = make_heading_re(chapter_prefix)
        nodes: list[Node] = []
        node_lookup: dict[str, Node] = {}
        current = None
        in_anexo = None
        in_glossario = False

        def flush(n):
            if n is not None:
                n.content = n.content.strip()
                if n.depth is not None and not n.chapter_number.startswith(("ANEXO", "GLOSSARIO")):
                    if is_section_header(n.content):
                        n.title = n.content
                        n.content = ""
                nodes.append(n)
                node_lookup[n.node_id] = n

        doc_prefix_id = document_id.replace("-", "")
        for line in lines:
            anexo_m = ROMAN_ANEXO_RE.match(line)
            if anexo_m:
                flush(current)
                current = None
                roman = anexo_m.group(1)
                node_id = f"{doc_prefix_id}_ANEXO_{roman}"
                n = Node(
                    node_id=node_id,
                    document_id=document_id,
                    document_type="NR",
                    chapter_number=f"ANEXO {roman}",
                    depth=0,
                    chapter_path=f"ANEXO {roman}",
                    title=None,
                    content=line,
                    has_table=False,
                )
                current = n
                in_anexo = roman
                in_glossario = False
                continue

            if GLOSSARIO_RE.match(line):
                flush(current)
                node_id = f"{doc_prefix_id}_GLOSSARIO"
                current = Node(
                    node_id=node_id,
                    document_id=document_id,
                    document_type="NR",
                    chapter_number="GLOSSARIO",
                    depth=0,
                    chapter_path="GLOSSARIO",
                    title="Glossário",
                    content="",
                    has_table=False,
                )
                in_glossario = True
                in_anexo = None
                continue

            if not in_anexo and not in_glossario:
                m = heading_re.match(line)
                candidate_node_id = f"{doc_prefix_id}_{m.group(1)}" if m else None
                if m and (candidate_node_id in node_lookup or (current is not None and m.group(1) == current.chapter_number)):
                    current = current
                    m = None
                if m:
                    flush(current)
                    chapter_number = m.group(1)
                    rest = m.group(2)
                    depth = chapter_number.count(".")
                    path_parts = chapter_number.split(".")
                    chapter_path = " > ".join(
                        ".".join(path_parts[: i + 1]) for i in range(len(path_parts))
                    )
                    node_id = f"{doc_prefix_id}_{chapter_number}"
                    current = Node(
                        node_id=node_id, document_id=document_id, document_type="NR",
                        chapter_number=chapter_number, depth=depth, chapter_path=chapter_path,
                        title=None, content=rest, has_table=False,
                    )
                    continue

            if current is not None:
                if "Tabela" in line:
                    current.has_table = True
                sep = " " if current.content and not current.content.endswith(("\n",)) else ""
                current.content = (current.content + sep + line).strip()

        flush(current)

        for n in nodes:
            n.references = extract_references(n.content, document_id, node_lookup)
        return nodes

    # ================= PROCESSAR CADA ARQUIVO =================
    generated_files = []

    for filename in arquivos_pdf:
        print(f"\n🔍 Processando: {filename}...")

        # Identificação inteligente de ID e Prefixo com base no nome do arquivo
        match = re.search(r'(?:NR|nr)\s*[-_]?\s*(\d+)', filename, re.IGNORECASE)
        if match:
            nr_number = match.group(1)
            document_id = f"NR-{nr_number}"
            chapter_prefix = nr_number
        else:
            digits = re.findall(r'\d+', filename)
            if digits:
                nr_number = digits[0]
                document_id = f"NR-{nr_number}"
                chapter_prefix = nr_number
            else:
                name_without_ext = os.path.splitext(filename)[0]
                document_id = name_without_ext.upper()
                chapter_prefix = "1"

        print(f"   -> Configurado como: ID='{document_id}', Prefixo='{chapter_prefix}'")

        try:
            nodes = parse_nr(filename, document_id, chapter_prefix)

            out = {
                "_descricao": f"Árvore extraída automaticamente de {document_id} via parser (nenhuma relação foi inferida manualmente).",
                "_total_nodes": len(nodes),
                "nodes": [asdict(n) for n in nodes],
            }

            out_filename = f"{document_id.replace('-', '_').lower()}_tree.json"
            with open(out_filename, "w", encoding="utf-8") as f:
                json.dump(out, f, ensure_ascii=False, indent=2)

            print(f"   ✅ Extraído com sucesso! Gerado: {out_filename} ({len(nodes)} nós)")
            generated_files.append(out_filename)
        except Exception as e:
            print(f"   ❌ Erro ao processar este arquivo: {e}")

    # ================= DOWNLOADS E ZIP =================
    if generated_files:
        if len(generated_files) > 1:
            zip_filename = "nrs_extraidas.zip"
            with zipfile.ZipFile(zip_filename, 'w') as zipf:
                for f in generated_files:
                    zipf.write(f)
            print(f"\n📦 Todos os JSONs foram compactados em: {zip_filename}")
            files.download(zip_filename)
            print("📥 Download do ZIP iniciado!")
        else:
            files.download(generated_files[0])
            print("\n📥 Download do JSON iniciado!")


📂 Encontrado(s) 10 arquivo(s) para processar: ['NR-05.pdf', 'NR-06.pdf', 'NR-10.pdf', 'NR-11.pdf', 'NR-12.pdf', 'NR-13.pdf', 'NR-20.pdf', 'NR-23.pdf', 'NR-33.pdf', 'NR-35.pdf']

🔍 Processando: NR-05.pdf...
   -> Configurado como: ID='NR-05', Prefixo='05'
   ✅ Extraído com sucesso! Gerado: nr_05_tree.json (1 nós)

🔍 Processando: NR-06.pdf...
   -> Configurado como: ID='NR-06', Prefixo='06'
   ✅ Extraído com sucesso! Gerado: nr_06_tree.json (3 nós)

🔍 Processando: NR-10.pdf...
   -> Configurado como: ID='NR-10', Prefixo='10'
   ✅ Extraído com sucesso! Gerado: nr_10_tree.json (118 nós)

🔍 Processando: NR-11.pdf...
   -> Configurado como: ID='NR-11', Prefixo='11'
   ✅ Extraído com sucesso! Gerado: nr_11_tree.json (38 nós)

🔍 Processando: NR-12.pdf...
   -> Configurado como: ID='NR-12', Prefixo='12'
   ✅ Extraído com sucesso! Gerado: nr_12_tree.json (235 nós)

🔍 Processando: NR-13.pdf...
   -> Configurado como: ID='NR-13', Prefixo='13'
   ✅ Extraído com sucesso! Gerado: nr_13_tree.json (165

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download do ZIP iniciado!
